# FraudShield: Exploratory Data Analysis (EDA)
### Intelligent Fraud Detection and Risk Analysis System

**Objectives:**
1. Inspect class distribution and quantify the extreme class imbalance (~0.17% fraud).
2. Profile transaction `Amount` and `Time` distributions to determine scaling strategy.
3. Analyze correlation of PCA features (`V1` to `V28`) with the target `Class`.
4. Establish preprocessing requirements to avoid data leakage before model training.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

data_path = os.path.join("..", "data", "raw", "creditcard.csv")
if not os.path.exists(data_path):
    from data.sample_data_generator import ensure_dataset_available
    data_path = ensure_dataset_available()

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

## 1. Class Distribution Analysis
In credit card fraud detection, accuracy is misleading because legitimate transactions overwhelmingly dominate.

In [ ]:
class_counts = df["Class"].value_counts()
fraud_pct = (class_counts[1] / len(df)) * 100

print(f"Legitimate Transactions (0): {class_counts[0]:,} ({(100 - fraud_pct):.3f}%)")
print(f"Fraudulent Transactions (1): {class_counts[1]:,} ({fraud_pct:.3f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette=["#2b5c8f", "#d9534f"], ax=ax)
ax.set_title("Transaction Class Imbalance (0: Legit, 1: Fraud)", fontweight="bold")
ax.set_yscale("log")
ax.set_ylabel("Count (Log Scale)")
plt.show()

## 2. Transaction Amount Distribution & Outliers
Transaction amount has extreme positive skew (most transactions are small, with rare huge amounts). We use `RobustScaler` (median & IQR) rather than `StandardScaler`.

In [ ]:
print("Legitimate Transactions Amount Statistics:")
print(df[df["Class"] == 0]["Amount"].describe())

print("\nFraudulent Transactions Amount Statistics:")
print(df[df["Class"] == 1]["Amount"].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df[df["Class"] == 0]["Amount"], bins=50, ax=axes[0], color="#2b5c8f", kde=True)
axes[0].set_title("Legitimate Transaction Amounts ($)")
axes[0].set_xlim(0, 1000)

sns.histplot(df[df["Class"] == 1]["Amount"], bins=50, ax=axes[1], color="#d9534f", kde=True)
axes[1].set_title("Fraudulent Transaction Amounts ($)")
axes[1].set_xlim(0, 1000)
plt.tight_layout()
plt.show()

## 3. Correlation with Target Class
Identify which PCA features are most negatively or positively correlated with fraud.

In [ ]:
correlations = df.corr()["Class"].sort_values()

plt.figure(figsize=(12, 6))
correlations[:-1].plot(kind="bar", color="#4a7c59")
plt.title("Feature Correlation with Target 'Class'", fontweight="bold")
plt.ylabel("Pearson Correlation Coefficient")
plt.axhline(0, color="black", lw=0.8)
plt.tight_layout()
plt.show()

print("Top 5 Positive Indicators:", correlations.tail(6)[:-1].to_dict())
print("Top 5 Negative Indicators:", correlations.head(5).to_dict())

## 4. Key Takeaways & Preprocessing Directives
1. **Class Imbalance**: Accuracy cannot be used. We must report Precision, Recall, F1-Score, and PR-AUC.
2. **Data Leakage Prevention**: Stratified train/test split must be applied before fitting scalers.
3. **Feature Scaling**: RobustScaler on `Amount` and `Time`; passthrough on `V1`–`V28`.
4. **Models**: Logistic Regression with `class_weight='balanced'` as the baseline, followed by Random Forest Classifier.